# Entraînement du Transformer JAX/Flax pour PowerCast
Ce notebook est conçu pour être exécuté sur **Google Colab** avec un accélérateur GPU (T4).

## 1. Installation des dépendances
Exécutez cette cellule pour installer Flax, Optax et PyTorch (pour les Dataloaders).

In [ ]:
!pip install -q jax jaxlib flax optax
!pip install -q scikit-learn pandas matplotlib seaborn

## 2. Téléchargement et Préparation des données

In [ ]:
import os
import requests
import pandas as pd
import numpy as np

if not os.path.exists('eco2mix_cleaned.csv'):
    print('Téléchargement des données brutes...')
    url = 'https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/eco2mix-national-cons-def/exports/csv?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B'
    response = requests.get(url, stream=True)
    with open('eco2mix_raw.csv', 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print('Nettoyage rapide...')
    df = pd.read_csv('eco2mix_raw.csv', sep=';', low_memory=False)
    df.rename(columns={'Date et Heure': 'datetime', 'Consommation (MW)': 'consumption'}, inplace=True)
    df.dropna(subset=['datetime'], inplace=True)
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    df.sort_values('datetime', inplace=True)
    df.set_index('datetime', inplace=True)
    df = df[~df.index.duplicated(keep='first')]
    df = df.resample('30min').asfreq()
    df['consumption'] = df['consumption'].interpolate(method='time')
    df.reset_index(inplace=True)
    df.to_csv('eco2mix_cleaned.csv', index=False)
    print('Données prêtes !')

## 3. Architecture Transformer (JAX/Flax) et Entraînement

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from flax.training.train_state import TrainState
import optax
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import time

print("JAX Devices:", jax.devices())

class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_length, pred_length):
        self.data = data
        self.seq_length = seq_length
        self.pred_length = pred_length
        
    def __len__(self):
        return len(self.data) - self.seq_length - self.pred_length + 1
        
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_length]
        y = self.data[idx + self.seq_length : idx + self.seq_length + self.pred_length]
        return torch.FloatTensor(x).unsqueeze(-1), torch.FloatTensor(y)

class PositionalEncoding(nn.Module):
    d_model: int
    max_len: int = 5000

    @nn.compact
    def __call__(self, x):
        pe = np.zeros((self.max_len, self.d_model))
        position = np.arange(0, self.max_len, dtype=np.float32)[:, np.newaxis]
        div_term = np.exp(np.arange(0, self.d_model, 2, dtype=np.float32) * -(np.log(10000.0) / self.d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        seq_len = x.shape[1]
        pe_slice = jnp.array(pe[:seq_len, :])
        return x + pe_slice

class TransformerEncoderBlock(nn.Module):
    num_heads: int
    mlp_dim: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, inputs, deterministic=True):
        x = nn.LayerNorm()(inputs)
        x = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            dropout_rate=self.dropout_rate,
            qkv_features=inputs.shape[-1] // self.num_heads
        )(x, x, deterministic=deterministic)
        x = x + inputs
        
        y = nn.LayerNorm()(x)
        y = nn.Dense(self.mlp_dim)(y)
        y = nn.relu(y)
        y = nn.Dropout(rate=self.dropout_rate)(y, deterministic=deterministic)
        y = nn.Dense(inputs.shape[-1])(y)
        y = y + x
        return y

class TimeSeriesTransformer(nn.Module):
    d_model: int = 32
    num_heads: int = 4
    num_layers: int = 2
    mlp_dim: int = 64
    dropout_rate: float = 0.1
    pred_len: int = 1

    @nn.compact
    def __call__(self, x, deterministic=True):
        x = nn.Dense(self.d_model)(x)
        x = PositionalEncoding(d_model=self.d_model)(x)
        
        for _ in range(self.num_layers):
            x = TransformerEncoderBlock(
                num_heads=self.num_heads,
                mlp_dim=self.mlp_dim,
                dropout_rate=self.dropout_rate
            )(x, deterministic=deterministic)
            
        x = x[:, -1, :]
        x = nn.LayerNorm()(x)
        x = nn.Dense(self.pred_len)(x)
        return x

# Préparation des données
df = pd.read_csv('eco2mix_cleaned.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
last_date = df['datetime'].max()
test_start = last_date - pd.Timedelta(days=30)
train_start = test_start - pd.Timedelta(days=365 * 2)

df = df[(df['datetime'] >= train_start) & (df['datetime'] <= last_date)].copy()
values = df['consumption'].values
train_mask = df['datetime'] < test_start
train_values = values[train_mask]
test_values = values[~train_mask]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_values.reshape(-1, 1)).flatten()
seq_length = 168
pred_length = 1
test_context = np.concatenate([train_values[-seq_length:], test_values])
test_scaled = scaler.transform(test_context.reshape(-1, 1)).flatten()

train_dataset = TimeSeriesDataset(train_scaled, seq_length, pred_length)
test_dataset = TimeSeriesDataset(test_scaled, seq_length, pred_length)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Modèle & TrainState
model = TimeSeriesTransformer()
rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)
dummy_x = jnp.ones((1, seq_length, 1))
variables = model.init(init_rng, dummy_x, deterministic=True)

tx = optax.adam(learning_rate=0.005)
state = TrainState.create(apply_fn=model.apply, params=variables['params'], tx=tx)

@jax.jit
def train_step(state, batch_x, batch_y, dropout_rng):
    def loss_fn(params):
        preds = state.apply_fn({'params': params}, batch_x, deterministic=False, rngs={'dropout': dropout_rng})
        loss = jnp.mean(optax.l2_loss(preds, batch_y))
        return loss, preds
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, preds), grads = grad_fn(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

@jax.jit
def eval_step(state, batch_x):
    return state.apply_fn({'params': state.params}, batch_x, deterministic=True)

epochs = 5
for epoch in range(epochs):
    rng, dropout_rng = jax.random.split(rng)
    total_loss = 0.0
    start_time = time.time()
    for x_batch, y_batch in train_loader:
        x_b = jnp.array(x_batch.numpy())
        y_b = jnp.array(y_batch.numpy())
        rng, dropout_batch_rng = jax.random.split(rng)
        state, loss = train_step(state, x_b, y_b, dropout_batch_rng)
        total_loss += loss
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f} - {elapsed:.1f}s")

# Évaluation
predictions, actuals = [], []
for x_batch, y_batch in test_loader:
    x_b = jnp.array(x_batch.numpy())
    preds = eval_step(state, x_b)
    predictions.append(np.array(preds))
    actuals.append(y_batch.numpy())

predictions = np.concatenate(predictions, axis=0)
actuals = np.concatenate(actuals, axis=0)
predictions = scaler.inverse_transform(predictions).flatten()
actuals = scaler.inverse_transform(actuals).flatten()

metrics = {
    'RMSE': float(np.sqrt(mean_squared_error(actuals, predictions))),
    'MAE': mean_absolute_error(actuals, predictions),
    'MAPE': mean_absolute_percentage_error(actuals, predictions) * 100
}
print("\n--- Métriques Transformer ---")
for k, v in metrics.items():
    print(f"{k}: {v:.2f}")

plt.figure(figsize=(15, 6))
time_axis = df['datetime'][~train_mask].values
plt.plot(time_axis, actuals, label='Vérité terrain')
plt.plot(time_axis, predictions, label='Transformer (Prédiction)', alpha=0.7)
plt.title(f"Prédiction Transformer - Derniers 30 jours (MAPE: {metrics['MAPE']:.2f}%)")
plt.legend()
plt.show()